**What is an "agent"?**

Normally, an LLM just reads your text and writes text back. That's it.

An **agent** is an LLM that can also **decide to use a tool** (a small piece of code you
wrote, like a calculator function) when it needs to, and then use the tool's answer to
write a better final response.

In [ ]:
!pip install google-genai python-dotenv

In [2]:
import os
from google import genai
from google.genai import types
from dotenv import load_dotenv

load_dotenv()
api_key = os.getenv("GEMINI_API_KEY")

if not api_key:
    raise SystemExit("No GEMINI_API_KEY found. Add it to a .env file before running this notebook.")

client = genai.Client(api_key=api_key)
print("Client is ready.")

Client is ready.


## Step 1: Define our tools(python function)

In [ ]:
def calculator(operation: str, a: float, b: float) -> float:
    """
    Performs a basic math operation on two numbers.
    operation: "add", "subtract", "multiply", "divide"
    a: the first number
    b: the second number
    """
    if operation == "add":
        return a + b
    elif operation == "subtract":
        return a - b
    elif operation == "multiply":
        return a * b
    elif operation == "divide":
        if b == 0:
            raise ValueError("Cannot divide by zero.")
        return a / b
    else:
        raise ValueError(f"Unknown operation: {operation}")

def lookup_word_meaning(word: str) -> str:
    """
    Looks up the meaning of a word from a small glossary.
    """
    glossary = {
        "agent": "An AI system that can decide to use tools or take actions, not just reply with text.",
        "embedding": "A list of numbers that represents the meaning of a piece of text.",
        "rag": "Retrieval-Augmented Generation: looking up relevant text first, then answering with it.",
        "prompt": "The text instructions or question you give to an LLM."
    }

    word = word.lower().strip()

    if word in glossary:
        return glossary[word]

    return f"Sorry, '{word}' is not in this small glossary."

## Step 2: Let Gemini use these tools

We just put our functions in a list called `tools`. Gemini reads their docstrings and type
hints, decides on its own if a question needs one of them, calls the function for us, and
then writes a final answer using the result. This is called **automatic function calling**.

In [4]:
def ask_agent(question):
    """
    Sends a question to Gemini, letting it call our tools if it needs to.
    """
    try:
        response = client.models.generate_content(
            model="gemini-3.6-flash",
            contents=question,
            config=types.GenerateContentConfig(
                tools=[calculator, lookup_word_meaning]
            )
        )
        return response.text
    except Exception as error:
        return f"Something went wrong: {error}"

## Step 3: Test questions

1. One that needs the **calculator** tool.
2. One that needs the **lookup** tool.
3. One that needs **no tool at all** 

In [5]:
test_questions = [
    "What is 482 multiplied by 17?",
    "What does the word 'embedding' mean?",
    "Say hello in French."
]

for question in test_questions:
    print("=" * 70)
    print("QUESTION:", question)
    print("ANSWER:", ask_agent(question))

Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


QUESTION: What is 482 multiplied by 17?
ANSWER: 482 multiplied by 17 is 8,194.
QUESTION: What does the word 'embedding' mean?
ANSWER: An **embedding** is a list of numbers (a vector) that represents the meaning or semantic content of a piece of text, allowing computers to process and compare text mathematically.
QUESTION: Say hello in French.
ANSWER: "Hello" in French is **"Bonjour"** (formal/general) or **"Salut"** (informal).
